## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

In [ ]:
# OpenAI Agents SDK lives in the PyPI package `openai-agents` (import name: agents).
%pip install -q "openai-agents>=0.0.17" pydantic python-dotenv sendgrid certifi tiktoken "gradio>=5"

import gradio as gr
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool, RunResult
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown


In [ ]:
import certifi

load_dotenv(override=True)

# SendGrid uses urllib; many macOS / Jupyter setups fail verification without an explicit CA bundle.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

### Guardrails & token budget

- **Input guardrails** (tripwires): non-empty prompt, max size, simple injection heuristics.
- **Output guardrails**: writer report length + risky markup; email agent output check; manager output size cap.
- **`RESEARCH_RUN_CONFIG`**: `call_model_input_filter` trims long **user** messages and **tool outputs** (skips compact planner JSON) using **tiktoken** to cap tokens and **reduce cost**.

All `Runner.run` calls in this notebook pass `run_config=RESEARCH_RUN_CONFIG` so the filter applies consistently.


In [ ]:
import copy

import tiktoken
from agents import input_guardrail, output_guardrail, GuardrailFunctionOutput, RunConfig
from agents.run_config import CallModelData, ModelInputData

# --- Token budget: applied on every model call via RunConfig (shrinks context, lowers cost) ---
_TIK = tiktoken.get_encoding("cl100k_base")
MAX_TOOL_OUTPUT_TOKENS = 3500
MAX_USER_MESSAGE_TOKENS = 2500
_PLAN_JSON_MARKER = '"searches"'


def _clip_tokens(text: str, max_tokens: int) -> tuple[str, int]:
    ids = _TIK.encode(text)
    if len(ids) <= max_tokens:
        return text, 0
    clipped = _TIK.decode(ids[:max_tokens])
    return clipped + f"\n\n[Truncated ~{len(ids) - max_tokens} tokens to reduce API cost]\n", len(ids) - max_tokens


def _budget_trim_content(content, max_tokens: int) -> None:
    if content is None:
        return
    if isinstance(content, str):
        return
    if isinstance(content, list):
        for part in content:
            if isinstance(part, dict):
                txt = part.get("text")
                if isinstance(txt, str):
                    part["text"], _ = _clip_tokens(txt, max_tokens)


def research_call_model_input_filter(data: CallModelData) -> ModelInputData:
    """Called by the SDK immediately before each model request; trim oversized strings."""
    payload = copy.deepcopy(data.model_data)
    for item in payload.input:
        if not isinstance(item, dict):
            continue
        if item.get("type") == "function_call_output" and isinstance(item.get("output"), str):
            out = item["output"]
            if _PLAN_JSON_MARKER in out[:1200]:
                continue
            item["output"], _ = _clip_tokens(out, MAX_TOOL_OUTPUT_TOKENS)
            continue
        if item.get("role") in ("user", "developer", "system"):
            _budget_trim_content(item.get("content"), MAX_USER_MESSAGE_TOKENS)
    return payload


RESEARCH_RUN_CONFIG = RunConfig(call_model_input_filter=research_call_model_input_filter)


def _flatten_user_text(inp) -> str:
    if isinstance(inp, str):
        return inp
    parts: list[str] = []
    if isinstance(inp, list):
        for it in inp:
            if isinstance(it, dict):
                c = it.get("content")
                if isinstance(c, str):
                    parts.append(c)
                elif isinstance(c, list):
                    for p in c:
                        if isinstance(p, dict) and isinstance(p.get("text"), str):
                            parts.append(p["text"])
    return "\n".join(parts)


MAX_FLAT_INPUT_CHARS = 120_000
INJECTION_MARKERS = (
    "ignore previous instructions",
    "ignore all previous",
    "system prompt",
    "reveal your api key",
    "sendgrid api key",
)


@input_guardrail(name="nonempty_research_input", run_in_parallel=False)
def nonempty_research_input(ctx, agent, message):
    text = _flatten_user_text(message).strip()
    if len(text) < 3:
        return GuardrailFunctionOutput(output_info="Input too short", tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)


@input_guardrail(name="oversized_prompt_guard", run_in_parallel=False)
def oversized_prompt_guard(ctx, agent, message):
    text = _flatten_user_text(message)
    if len(text) > MAX_FLAT_INPUT_CHARS:
        return GuardrailFunctionOutput(
            output_info=f"Input exceeds {MAX_FLAT_INPUT_CHARS} characters.",
            tripwire_triggered=True,
        )
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)


@input_guardrail(name="basic_injection_heuristic", run_in_parallel=False)
def basic_injection_heuristic(ctx, agent, message):
    low = _flatten_user_text(message).lower()
    hit = next((m for m in INJECTION_MARKERS if m in low), None)
    if hit:
        return GuardrailFunctionOutput(output_info=f"Flagged phrase: {hit!r}", tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)


SHARED_INPUT_GUARDRAILS = [
    nonempty_research_input,
    oversized_prompt_guard,
    basic_injection_heuristic,
]


@output_guardrail(name="writer_report_sanity")
def writer_report_sanity(ctx, agent, output):
    md = getattr(output, "markdown_report", None)
    if not isinstance(md, str) or len(md.strip()) < 200:
        return GuardrailFunctionOutput(output_info="Report missing or too short", tripwire_triggered=True)
    low = md.lower()
    if "<script" in low or "javascript:" in low or "onerror=" in low:
        return GuardrailFunctionOutput(output_info="Suspicious markup in report", tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)


@output_guardrail(name="email_response_sanity")
def email_response_sanity(ctx, agent, output):
    low = str(output).lower()
    if "<script" in low or "javascript:" in low:
        return GuardrailFunctionOutput(output_info="Suspicious content in email agent output", tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)


@output_guardrail(name="manager_output_size")
def manager_output_size(ctx, agent, output):
    if len(str(output)) > 600_000:
        return GuardrailFunctionOutput(output_info="Manager output too large", tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)



In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message, run_config=RESEARCH_RUN_CONFIG)

display(Markdown(result.final_output))


### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"""You are a helpful research assistant. You will receive:
- ORIGINAL QUERY — what the user wants to know
- CLARIFYING Q&A — three questions and answers that narrow audience, timeframe, depth, and angle

Using that context, propose exactly {HOW_MANY_SEARCHES} complementary web searches. Each search should reflect
the clarified scope (not generic restatements of the title). Explain briefly why each search matters."""

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:
message = """Latest AI Agent frameworks in 2026

CLARIFYING Q&A:
Q1: What depth: landscape survey or deep comparison of one stack?
A1: Landscape survey for practitioners evaluating options.
Q2: Target audience and deployment context?
A2: Application developers and ML engineers; cloud-agnostic.
Q3: Time window for "current" frameworks?
A3: 2024–2025 releases and adoption signals."""

with trace("Search"):
    result = await Runner.run(planner_agent, message, run_config=RESEARCH_RUN_CONFIG)
    print(result.final_output)


In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send HTML email via SendGrid. Catches API/network failures so the agent can recover."""
    try:
        sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
        from_email = Email("calistus@igwilo.com")  # Change to your verified sender
        to_email = To("ebunilo@yahoo.com")  # Change to your recipient
        content = Content("text/html", html_body)
        mail = Mail(from_email, to_email, subject, content).get()
        sg.client.mail.send.post(request_body=mail)
        return {"status": "success"}
    except Exception as exc:
        return {"status": "error", "message": str(exc)}

In [ ]:
send_email

In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
    handoff_description="Send the finalized research report: convert markdown to HTML and email it with a strong subject line.",
    output_guardrails=[email_response_sanity],
)


In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
    output_guardrails=[writer_report_sanity],
)


### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [ ]:
async def plan_searches(query: str, clarification_block: str = ""):
    """Use the planner_agent to plan searches. Pass optional CLARIFYING Q&A text to tune the plan."""
    print("Planning searches...")
    block = clarification_block.strip() or "None provided — infer reasonable scope from the query alone."
    payload = f"ORIGINAL QUERY:\n{query}\n\nCLARIFYING Q&A:\n{block}"
    result = await Runner.run(planner_agent, payload, run_config=RESEARCH_RUN_CONFIG)
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input, run_config=RESEARCH_RUN_CONFIG)
    return result.final_output


### The next 2 functions write a report and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input, run_config=RESEARCH_RUN_CONFIG)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report, run_config=RESEARCH_RUN_CONFIG)
    print("Email sent")
    return report


### Autonomous Research Manager (after *your* clarifying answers)

**Showtime** embeds a small **Gradio** UI in the notebook: generate three questions, type answers in the form, then run **`research_manager`** (plan → search → write → email handoff). No separate `.py` file—everything runs from the Showtime cell.

The manager uses **agents-as-tools** (`as_tool`) for plan → search → write and a **handoff** to the email agent.

In [ ]:
class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Exactly three clarifying questions that narrow scope, audience, timeframe, and depth.",
    )


clarification_question_agent = Agent(
    name="ClarificationQuestionAgent",
    instructions=(
        "Given the user's research request, output exactly THREE clarifying questions that would sharpen web search. "
        "Cover disambiguation where helpful: audience, time range, depth (survey vs deep dive), geography or vendor focus, "
        "and what a good answer must include."
    ),
    model="gpt-4o-mini",
    model_settings=ModelSettings(temperature=0.3, max_tokens=800),
    output_type=ClarifyingQuestions,
    input_guardrails=SHARED_INPUT_GUARDRAILS,
)


async def _dump_model_json(result: RunResult) -> str:
    return result.final_output.model_dump_json()


async def _writer_markdown_only(result: RunResult) -> str:
    return result.final_output.markdown_report


plan_searches_tool = planner_agent.as_tool(
    tool_name="plan_web_searches",
    tool_description=(
        "First step: pass one string that already contains ORIGINAL QUERY and CLARIFYING Q&A (human answers from the notebook prompts). "
        f"Returns JSON WebSearchPlan with exactly {HOW_MANY_SEARCHES} searches tuned to that brief."
    ),
    custom_output_extractor=_dump_model_json,
)

write_report_tool = writer_agent.as_tool(
    tool_name="write_research_report",
    tool_description=(
        "Third step: pass one string with original query, clarifying Q&A, and summarized search results. "
        "Returns the final markdown report body."
    ),
    custom_output_extractor=_writer_markdown_only,
)


@function_tool
async def execute_search_plan(search_plan_json: str) -> str:
    """Second step: run each search; tolerate individual failures so the manager can continue."""
    plan = WebSearchPlan.model_validate_json(search_plan_json)
    chunks: list[str] = []
    for item in plan.searches:
        try:
            chunks.append(await search(item))
        except Exception as exc:
            chunks.append(f"[Search failed for {item.query!r}: {exc}]")
    return "\n\n---\n\n".join(chunks)


MANAGER_INSTRUCTIONS = """
You are the Deep Research Manager. The human has already answered clarifying questions in the notebook; the user message contains
ORIGINAL QUERY and CLARIFYING Q&A. Use tools in order, then hand off for email.

1) Call plan_web_searches with one `input` string that matches the user message (ORIGINAL QUERY + CLARIFYING Q&A).
2) Call execute_search_plan with the exact JSON string returned from plan_web_searches (no markdown fences).
3) Call write_research_report with one `input` string containing: the original query, the CLARIFYING Q&A block, and the text from execute_search_plan.
4) Hand off to the Email agent. Your handoff message must include the full markdown report returned in step 3.

If a tool fails, retry once with corrected input.
"""

research_manager = Agent(
    name="DeepResearchManager",
    instructions=MANAGER_INSTRUCTIONS.strip(),
    tools=[
        plan_searches_tool,
        execute_search_plan,
        write_report_tool,
    ],
    handoffs=[email_agent],
    model="gpt-4o-mini",
    model_settings=ModelSettings(temperature=0.4, top_p=0.9, max_tokens=8192),
    input_guardrails=SHARED_INPUT_GUARDRAILS,
    output_guardrails=[manager_output_size],
)


### Showtime — Gradio UI (inline), then manager

Run the cell below. Use **1. Generate clarifying questions**, fill the three answer boxes, then **2. Run research manager**. Re-run the cell if you need a fresh UI.

A **commented** legacy linear pipeline (`input()` / no Gradio) is at the bottom of the code cell for debugging.


In [ ]:
def _questions_markdown(qs: list[str]) -> str:
    if not qs:
        return "*Click **1. Generate clarifying questions** first.*"
    lines = ["### Clarifying questions", ""]
    for i, q in enumerate(qs, 1):
        lines.append(f"**Q{i}:** {q}")
    return "\n\n".join(lines)


async def _gradio_generate(topic: str):
    topic = (topic or "").strip()
    if not topic:
        return "Enter a research topic above.", [], gr.update(visible=False), gr.update(visible=False), "", "", ""
    with trace("Deep research — clarifying questions (Gradio)"):
        cq_result = await Runner.run(
            clarification_question_agent, topic, run_config=RESEARCH_RUN_CONFIG
        )
    qs = list(cq_result.final_output.questions)
    if not qs:
        return "No questions returned.", [], gr.update(visible=False), gr.update(visible=False), "", "", ""
    return (
        _questions_markdown(qs),
        qs,
        gr.update(visible=True),
        gr.update(visible=True),
        "",
        "",
        "",
    )


async def _gradio_run_manager(topic: str, qs: list, a1: str, a2: str, a3: str):
    if not qs:
        return "Generate clarifying questions first (step 1)."
    topic = (topic or "").strip()
    answers = [(a1 or "").strip(), (a2 or "").strip(), (a3 or "").strip()]
    while len(answers) < len(qs):
        answers.append("")
    clarification_block = "\n".join(
        f"Q{j}: {qs[j - 1]}\nA{j}: {answers[j - 1]}" for j in range(1, len(qs) + 1)
    )
    manager_payload = f"ORIGINAL QUERY:\n{topic}\n\nCLARIFYING Q&A:\n{clarification_block}"
    with trace("Deep research — manager (Gradio)"):
        mgr_result = await Runner.run(
            research_manager, manager_payload, run_config=RESEARCH_RUN_CONFIG
        )
    return f"**Manager finished.**\n\n{mgr_result.final_output}"


if "_GRADIO_CLARIFY_UI" in globals():
    try:
        globals()["_GRADIO_CLARIFY_UI"].close()
    except Exception:
        pass

with gr.Blocks(title="Deep research — clarify & run") as _clarify_ui:
    _q_state = gr.State([])
    _topic = gr.Textbox(label="Research topic", lines=2, value="Latest AI Agent frameworks in 2026")
    _b1 = gr.Button("1. Generate clarifying questions", variant="primary")
    _qmd = gr.Markdown()
    with gr.Group(visible=False) as _answer_panel:
        gr.Markdown("### Your answers")
        _a1 = gr.Textbox(label="Answer to Q1", lines=2)
        _a2 = gr.Textbox(label="Answer to Q2", lines=2)
        _a3 = gr.Textbox(label="Answer to Q3", lines=2)
    _b2 = gr.Button("2. Run research manager", variant="primary", visible=False)
    _out = gr.Markdown()

    _b1.click(
        _gradio_generate,
        [_topic],
        [_qmd, _q_state, _answer_panel, _b2, _a1, _a2, _a3],
    )
    _b2.click(
        _gradio_run_manager,
        [_topic, _q_state, _a1, _a2, _a3],
        [_out],
    )

_clarify_ui.queue()
_clarify_ui.launch(inline=True, height=720, prevent_thread_lock=True, quiet=True)
globals()["_GRADIO_CLARIFY_UI"] = _clarify_ui
